In [2]:
# ===== Summarize temperature=0.5 results: counts & percentages of A/B/C per LLM =====
# Looks for "0.5-*.csv" files (e.g., 0.5-GPT4o.csv, 0.5-Llama.csv, ...),
# parses the "answer" column (e.g., "A - because ..."), and reports counts & percents per model.
#
# Outputs:
#   - out/temp_0.5_choice_counts.csv
#   - out/temp_0.5_choice_counts_by_model.tex  (Overleaf-ready compact table)

import os, re, glob
import pandas as pd

BASE_DIR = "."           
PATTERN  = "1.0-*.csv"
OUT_DIR  = os.path.join(BASE_DIR, "out")
os.makedirs(OUT_DIR, exist_ok=True)

def model_name_from_path(p: str) -> str:
    fname = os.path.basename(p)
    # strip leading "0.5-" and trailing ".csv"
    return re.sub(r"^0\.5-|\.csv$", "", fname)

def extract_choice_letter(ans: str):
    """Return 'A','B','C' from an answer like 'Answer: B - ...' or 'C - ...'."""
    if not isinstance(ans, str) or not ans.strip():
        return None
    s = re.sub(r'^\s*Answer:\s*', '', ans, flags=re.IGNORECASE).strip()
    m = re.match(r'^\s*([ABC])\b', s, flags=re.IGNORECASE)
    if not m:
        m = re.search(r'\b([ABC])\b', s[:80], flags=re.IGNORECASE)
    return m.group(1).upper() if m else None

def load_and_count_choice(path: str) -> dict:
    df = pd.read_csv(path)
    if "answer" not in df.columns:
        raise ValueError(f"{path} is missing an 'answer' column.")
    choices = df["answer"].map(extract_choice_letter)
    valid = choices.dropna()
    counts = valid.value_counts()
    return {
        "A": int(counts.get("A", 0)),
        "B": int(counts.get("B", 0)),
        "C": int(counts.get("C", 0)),
        "total": int(valid.shape[0]),
        "invalid_rows": int(df.shape[0] - valid.shape[0]),
    }

# --- Discover files & build summary ---
files = sorted(glob.glob(os.path.join(BASE_DIR, PATTERN)))
if not files:
    raise SystemExit(f"No files matched {PATTERN} in {BASE_DIR}")

rows = []
for f in files:
    model = model_name_from_path(f)
    stats = load_and_count_choice(f)
    A, B, C, total = stats["A"], stats["B"], stats["C"], stats["total"]
    rows.append({
        "model": model,
        "A": A, "B": B, "C": C, "total": total,
        "pct_A": round(100.0 * A / total, 2) if total else 0.0,
        "pct_B": round(100.0 * B / total, 2) if total else 0.0,
        "pct_C": round(100.0 * C / total, 2) if total else 0.0,
        "invalid_rows": stats["invalid_rows"],
        "source_file": os.path.basename(f),
    })

summary = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)

# Save CSV
csv_path = os.path.join(OUT_DIR, "temp_0.5_choice_counts.csv")
summary.to_csv(csv_path, index=False)
print(f"[OK] Wrote {csv_path}")

# Print a clean view
print("\nCounts and percentages by model (temperature = 0.5):")
print(summary[["model","A","B","C","total","pct_A","pct_B","pct_C","invalid_rows"]].to_string(index=False))

# --- Overleaf-ready LaTeX table (compact headers) ---
try:
    from textwrap import dedent
    latex_cols = ["model","A","B","C","total","pct_A","pct_B","pct_C","invalid_rows"]
    latex_df = summary[latex_cols].copy()
    latex_df = latex_df.rename(columns={
        "pct_A": r"\makecell{\%\\A}",
        "pct_B": r"\makecell{\%\\B}",
        "pct_C": r"\makecell{\%\\C}",
        "invalid_rows": r"\makecell{invalid\\rows}"
    })
    tex_path = os.path.join(OUT_DIR, "temp_0.5_choice_counts_by_model.tex")
    with open(tex_path, "w", encoding="utf-8") as f:
        f.write(dedent(r"""
        % Requires: \usepackage{booktabs}
        % Optional for multi-line headers: \usepackage{makecell}
        """).lstrip())
        f.write(latex_df.to_latex(index=False, float_format="%.2f", escape=False))
    print(f"[OK] Wrote {tex_path}")
except Exception as e:
    print(f"[WARN] Could not write LaTeX table: {e}")

[OK] Wrote .\out\temp_0.5_choice_counts.csv

Counts and percentages by model (temperature = 0.5):
        model  A  B  C  total  pct_A  pct_B  pct_C  invalid_rows
1.0-GPT4o-ABC 12 21 87    120  10.00  17.50  72.50             0
1.0-Gemma-ABC 14 23 83    120  11.67  19.17  69.17             0
1.0-Llama-ABC 19 45 56    120  15.83  37.50  46.67             0
 1.0-Qwen-ABC  9 24 87    120   7.50  20.00  72.50             0
[OK] Wrote .\out\temp_0.5_choice_counts_by_model.tex
